In [ ]:
import os, shutil
import pandas as pd

In [ ]:
dialect = ["AAVE", "ChcE", "CollSgE", "IndE", "JamE"][0]

try:
    prompt_guard_toxic_results = pd.read_csv(f"./exp_image_level_promptguard/phase2_toxic_results_{dialect}/phase2_toxic_analyze.csv")
except:
    prompt_guard_toxic_results = None

try:
    prompt_guard_benign_results = pd.read_csv(f"./exp_image_level_promptguard/phase2_benign_results_{dialect}/phase2_benign_analyze.csv")
except:
    prompt_guard_benign_results = None

In [ ]:
candidates = prompt_guard_toxic_results[
    (prompt_guard_toxic_results["std_lpips_between_base_guard"] > 0.3) &
    (prompt_guard_toxic_results["dial_lpips_between_base_guard"] < 0.1) &
    (prompt_guard_toxic_results["std_base_q16"] > 0.7) &
    (prompt_guard_toxic_results["dial_base_q16"] > 0.7) &  # Both are similarly toxic
    (~prompt_guard_toxic_results["category"].isin(["Political_Figures", "Political_Metaphor",
                            "Explicit_Pornography", "Borderline_Pornography"]))
].copy()

# Sort by largest intervention difference between SAE and AAVE
candidates["intervention_gap"] = (
    candidates["std_lpips_between_base_guard"] -
    candidates["dial_lpips_between_base_guard"]
)

best_per_category = (
    candidates.sort_values("intervention_gap", ascending=False)
    .groupby("category")
    .first()
    .reset_index()
    .sort_values("intervention_gap", ascending=False)
).head(20)

print(best_per_category[["category",
                          "std_base_q16", "std_guard_q16",
                          "dial_base_q16", "dial_guard_q16",
                          "std_lpips_between_base_guard",
                          "dial_lpips_between_base_guard",
                          "std_noguard_img", "dial_noguard_img"]].to_string())

os.makedirs('./cherrypicking_candidate', exist_ok = True)

best_per_category.to_csv(f"./cherrypicking_candidate/promptguard_toxic_{dialect}.csv", index=False)

for _, row in best_per_category.iterrows():
    category = row["category"]
    std_noguard_img = row["std_noguard_img"]
    dial_noguard_img = row["dial_noguard_img"]

    std_noguard_img_path = f"./exp_image_level_promptguard/{std_noguard_img}"
    std_guarded_img_path = std_noguard_img_path.replace("noguard", "guarded")

    dial_noguard_img_path = f"./exp_image_level_promptguard/{dial_noguard_img}"
    dial_guarded_img_path = dial_noguard_img_path.replace("noguard", "guarded")

    os.makedirs(f"./cherrypicking_candidate/promptguard_toxic_{dialect}", exist_ok=True)

    shutil.copy(std_noguard_img_path, f"./cherrypicking_candidate/promptguard_toxic_{dialect}/{std_noguard_img_path.split('/')[-1]}")
    shutil.copy(std_guarded_img_path, f"./cherrypicking_candidate/promptguard_toxic_{dialect}/{std_guarded_img_path.split('/')[-1]}")
    shutil.copy(dial_noguard_img_path, f"./cherrypicking_candidate/promptguard_toxic_{dialect}/{dial_noguard_img_path.split('/')[-1]}")
    shutil.copy(dial_guarded_img_path, f"./cherrypicking_candidate/promptguard_toxic_{dialect}/{dial_guarded_img_path.split('/')[-1]}")



In [ ]:
benign_candidates = prompt_guard_benign_results[
    (prompt_guard_benign_results["std_lpips_between_base_guard"] > 0.3) &  # SAE: changed significantly
    (prompt_guard_benign_results["dial_lpips_between_base_guard"] < 0.05)  # AAVE: almost unchanged
].copy()
    
benign_candidates["intervention_gap"] = (
    benign_candidates["std_lpips_between_base_guard"] -
    benign_candidates["dial_lpips_between_base_guard"]
)

best_per_category = (
    benign_candidates.sort_values("intervention_gap", ascending=False)
    .groupby("category")
    .first()
    .reset_index()
    .sort_values("intervention_gap", ascending=False)
).head(20)

best_per_category.to_csv(f"./cherrypicking_candidate/promptguard_benign_{dialect}.csv", index=False)

print(f"Total candidates: {len(best_per_category)}")

for _, row in best_per_category.iterrows():
    category = row["category"]
    std_noguard_img = row["std_noguard_img"]
    dial_noguard_img = row["dial_noguard_img"]

    std_noguard_img_path = f"./exp_image_level_promptguard/{std_noguard_img}"
    std_guarded_img_path = std_noguard_img_path.replace("noguard", "guarded")

    dial_noguard_img_path = f"./exp_image_level_promptguard/{dial_noguard_img}"
    dial_guarded_img_path = dial_noguard_img_path.replace("noguard", "guarded")

    os.makedirs(f"./cherrypicking_candidate/promptguard_benign_{dialect}", exist_ok=True)

    shutil.copy(std_noguard_img_path, f"./cherrypicking_candidate/promptguard_benign_{dialect}/{std_noguard_img_path.split('/')[-1]}")
    shutil.copy(std_guarded_img_path, f"./cherrypicking_candidate/promptguard_benign_{dialect}/{std_guarded_img_path.split('/')[-1]}")
    shutil.copy(dial_noguard_img_path, f"./cherrypicking_candidate/promptguard_benign_{dialect}/{dial_noguard_img_path.split('/')[-1]}")
    shutil.copy(dial_guarded_img_path, f"./cherrypicking_candidate/promptguard_benign_{dialect}/{dial_guarded_img_path.split('/')[-1]}")
